# Statistical investigation

Explore descriptive relationships and quantify resampling uncertainty. All associations describe the synthetic process.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "deployment").exists():
    root = root.parent
snapshot = root / "deployment/snapshot"
profile = json.loads((snapshot / "data_report.json").read_text())
model = json.loads((snapshot / "model.json").read_text())
def sql(name):
    return pd.read_csv(snapshot / "analytics" / f"{name}.csv")


In [2]:
stats = profile["descriptive"]
print("Spearman(early severity, settlement days):", round(stats["severity_cycle_spearman"],3))
print("Actionable prevalence:", round(stats["eligible_prevalence"],4))
print("95% month-block bootstrap:", stats["monthly_bootstrap_prevalence_ci"])
print("Positive avoidable-cost quantiles:", stats["avoidable_quantiles_positive"])
display(sql("bottlenecks"))


Spearman(early severity, settlement days): 0.28
Actionable prevalence: 0.0652
95% month-block bootstrap: [0.06231444715612394, 0.06831869311159303]
Positive avoidable-cost quantiles: {'0.1': 102.86800000000001, '0.5': 330.39, '0.9': 1061.6520000000007, '0.99': 2499.688500000001}


,complexity,documented,claims,mean_days,p90_days
0,1,0,12647,30.345536,41.0
1,1,1,25440,26.817689,37.0
2,2,0,7415,34.979366,46.0
3,2,1,14844,31.399825,42.0
4,3,0,7541,37.553773,48.0
5,3,1,15367,33.827683,44.0
6,4,0,5812,39.670853,51.0
7,4,1,12015,35.976363,46.0
8,5,0,4531,43.196204,54.0
9,5,1,9519,39.284379,49.0


In [3]:
vendors = sql("vendors")
compare = vendors[["mean_paid","residual"]].agg(["min","max","std"])
display(compare)
display(vendors.sort_values("mean_paid",ascending=False).head()[["vendor_id","specialization","claims","mean_paid","residual"]])


,mean_paid,residual
min,1293.457955,-137.650042
max,5649.901767,134.663025
std,1421.398484,53.605459


,vendor_id,specialization,claims,mean_paid,residual
53,54,Structural,1143,5649.901767,130.228367
55,56,Structural,1218,5597.583366,54.495946
52,53,Structural,1229,5579.153059,70.594772
49,50,Structural,1119,5557.909902,76.272717
59,60,Structural,1140,5522.248825,-29.973994


Month-block resampling keeps within-month claims together and assumes those months are exchangeable for this summary. Residual costs subtract the mean for observed severity, complexity and coverage before aggregating by vendor. The descriptive adjustment provides no estimate of a causal vendor effect.